# Banking77 - Exploratory Data Analysis

Exploratory analysis of the Banking77 dataset: 77 intent classes for banking customer service queries.

**Goal:** Understand class distribution, text characteristics, and potential challenges before modeling.

In [ ]:
import sys

sys.path.insert(0, "../..")

from ml.utils.reproducibility import set_seed

set_seed(42)

from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from datasets import load_dataset

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (12, 5)

## 1. Load Data

In [ ]:
dataset = load_dataset("PolyAI/banking77")

train_df = pd.DataFrame(dataset["train"])
test_df = pd.DataFrame(dataset["test"])

# Get label names from the dataset features
label_names = dataset["train"].features["label"].names
train_df["label_name"] = train_df["label"].map(lambda x: label_names[x])
test_df["label_name"] = test_df["label"].map(lambda x: label_names[x])

print(f"Train samples: {len(train_df):,}")
print(f"Test samples:  {len(test_df):,}")
print(f"Total samples: {len(train_df) + len(test_df):,}")
print(f"\nColumns: {list(train_df.columns)}")
train_df.head()

## 2. Basic Statistics

In [ ]:
train_df["text_length"] = train_df["text"].str.len()
train_df["word_count"] = train_df["text"].str.split().str.len()

num_classes = train_df["label"].nunique()

print(f"Number of classes: {num_classes}")
print("\n--- Text Length (characters) ---")
print(f"  Mean: {train_df['text_length'].mean():.1f}")
print(f"  Min:  {train_df['text_length'].min()}")
print(f"  Max:  {train_df['text_length'].max()}")
print(f"  Std:  {train_df['text_length'].std():.1f}")
print("\n--- Word Count ---")
print(f"  Mean: {train_df['word_count'].mean():.1f}")
print(f"  Min:  {train_df['word_count'].min()}")
print(f"  Max:  {train_df['word_count'].max()}")
print("\n--- Samples per Class ---")
counts = train_df["label_name"].value_counts()
print(f"  Mean: {counts.mean():.1f}")
print(f"  Min:  {counts.min()} ({counts.idxmin()})")
print(f"  Max:  {counts.max()} ({counts.idxmax()})")
print(f"  Imbalance ratio (max/min): {counts.max() / counts.min():.2f}")

## 3. Class Distribution

In [ ]:
class_counts = train_df["label_name"].value_counts().sort_values(ascending=True)
mean_count = class_counts.mean()

fig, ax = plt.subplots(figsize=(14, 18))
colors = ["#e74c3c" if c < mean_count * 0.8 else "#3498db" for c in class_counts.values]
ax.barh(range(len(class_counts)), class_counts.values, color=colors)
ax.set_yticks(range(len(class_counts)))
ax.set_yticklabels(class_counts.index, fontsize=7)
ax.axvline(mean_count, color="black", linestyle="--", linewidth=0.8, label=f"Mean ({mean_count:.0f})")
ax.set_xlabel("Number of Samples")
ax.set_title("Class Distribution (red = below 80% of mean)")
ax.legend()
plt.tight_layout()
plt.show()

# Summary
below_threshold = (class_counts < mean_count * 0.8).sum()
print(f"\n{below_threshold} classes below 80% of mean count ({mean_count * 0.8:.0f} samples)")

## 4. Text Length Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(train_df["text_length"], bins=50, color="#3498db", edgecolor="white", alpha=0.8)
axes[0].axvline(
    train_df["text_length"].mean(), color="red", linestyle="--", label=f"Mean: {train_df['text_length'].mean():.0f}"
)
axes[0].set_xlabel("Character Length")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Text Length Distribution (Characters)")
axes[0].legend()

axes[1].hist(train_df["word_count"], bins=30, color="#2ecc71", edgecolor="white", alpha=0.8)
axes[1].axvline(
    train_df["word_count"].mean(), color="red", linestyle="--", label=f"Mean: {train_df['word_count'].mean():.0f}"
)
axes[1].set_xlabel("Word Count")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Text Length Distribution (Words)")
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Top Words

In [ ]:
# Stopwords to filter out
stopwords = {
    "i",
    "my",
    "me",
    "the",
    "a",
    "an",
    "is",
    "it",
    "to",
    "and",
    "of",
    "in",
    "for",
    "on",
    "was",
    "have",
    "has",
    "do",
    "does",
    "did",
    "can",
    "but",
    "not",
    "with",
    "this",
    "that",
    "be",
    "are",
    "from",
    "or",
    "at",
    "by",
    "you",
    "your",
    "how",
    "what",
    "why",
    "when",
    "where",
    "which",
    "who",
    "will",
    "would",
    "could",
    "should",
    "there",
    "been",
    "being",
    "if",
    "so",
    "than",
    "too",
    "very",
    "just",
    "about",
    "up",
    "out",
    "no",
    "get",
    "got",
    "am",
    "were",
    "had",
    "don't",
    "didn't",
    "won't",
    "can't",
    "t",
    "s",
    "m",
    "re",
    "ve",
    "d",
    "ll",
}

all_words = " ".join(train_df["text"].str.lower()).split()
filtered_words = [w for w in all_words if w not in stopwords and len(w) > 2]
word_freq = Counter(filtered_words).most_common(30)

words, counts = zip(*word_freq, strict=False)
fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(range(len(words)), counts, color="#8e44ad", edgecolor="white")
ax.set_xticks(range(len(words)))
ax.set_xticklabels(words, rotation=45, ha="right", fontsize=9)
ax.set_ylabel("Frequency")
ax.set_title("Top 30 Words (excluding stopwords)")
plt.tight_layout()
plt.show()

## 6. Class Overlap Analysis

Compute TF-IDF cosine similarity between class centroids to identify the most confusable class pairs.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Build one "document" per class by concatenating all texts
class_texts = train_df.groupby("label_name")["text"].apply(" ".join).reset_index()
class_texts.columns = ["label_name", "combined_text"]

vectorizer = TfidfVectorizer(max_features=5000, stop_words="english")
tfidf_matrix = vectorizer.fit_transform(class_texts["combined_text"])

sim_matrix = cosine_similarity(tfidf_matrix)
np.fill_diagonal(sim_matrix, 0)  # zero out self-similarity

# Find top 5 most similar pairs
names = class_texts["label_name"].values
pairs = []
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        pairs.append((names[i], names[j], sim_matrix[i, j]))

pairs.sort(key=lambda x: x[2], reverse=True)

print("Top 5 Most Similar Class Pairs (potential confusion):")
print("=" * 70)
for rank, (c1, c2, score) in enumerate(pairs[:5], 1):
    print(f"  {rank}. {c1}  <->  {c2}")
    print(f"     Cosine similarity: {score:.4f}")
    print()

In [ ]:
# Heatmap of top 15 most similar classes
top_pair_classes = set()
for c1, c2, _ in pairs[:10]:
    top_pair_classes.add(c1)
    top_pair_classes.add(c2)
top_pair_classes = sorted(top_pair_classes)[:15]

idx = [list(names).index(c) for c in top_pair_classes]
sub_matrix = sim_matrix[np.ix_(idx, idx)]

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    sub_matrix,
    xticklabels=top_pair_classes,
    yticklabels=top_pair_classes,
    cmap="YlOrRd",
    annot=True,
    fmt=".2f",
    ax=ax,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Cosine Similarity"},
)
ax.set_title("TF-IDF Cosine Similarity Between Most Overlapping Classes")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

## 7. Sample Inspection

Random examples from 5 different classes to build intuition about the data.

In [ ]:
import random

random.seed(42)

sample_classes = random.sample(sorted(train_df["label_name"].unique().tolist()), 5)

for cls in sample_classes:
    print(f"\n{'=' * 70}")
    print(f"Class: {cls}")
    print(f"{'=' * 70}")
    samples = train_df[train_df["label_name"] == cls].sample(3, random_state=42)
    for i, (_, row) in enumerate(samples.iterrows(), 1):
        print(f"  {i}. {row['text']}")

## 8. Summary of Key Findings

**Dataset overview:**
- Banking77 contains customer queries across 77 banking intent classes
- Relatively short texts (typical of chat/support queries)

**Key observations:**
1. **Class balance:** The dataset is relatively balanced, though some classes have notably fewer samples than others. Classes falling below 80% of the mean count may need attention during training (e.g., oversampling or class-weighted loss).

2. **Text characteristics:** Queries are short and conversational. Most are under ~20 words. This means models need to extract intent from minimal context.

3. **Vocabulary:** Domain-specific banking terms (card, transfer, payment, account) dominate. A domain-aware tokenizer or fine-tuned embeddings should help.

4. **Class overlap:** Several class pairs have high TF-IDF similarity, indicating they share vocabulary and may be hard to distinguish. These pairs are the primary source of misclassification and should be monitored closely during evaluation.

**Implications for modeling:**
- Short text length favors transformer-based models that capture contextual nuance
- Overlapping classes suggest fine-grained intent distinctions that benefit from pre-trained language models
- Consider hierarchical grouping or confusion-matrix analysis focused on the top similar pairs